# Celestial Mechanics and Star-Chart Predictors: Equations and Derivation

This notebook turns the planetary orbits and n-body gravity into a mechanics problem with generalized coordinates and a clear variational derivation.

## Background and physical setting
The system is studied through a small set of state variables that describe its configuration over time. In this notebook, the focus is on the relation between the observable behavior of the system and the underlying mechanical model.

## Why this model is useful
Using a Lagrangian or Hamiltonian description helps connect the physical intuition of the system to a compact set of equations. It also gives a natural way to compare the model with simulation outputs and to interpret stability, resonance, and energy exchange.

## Coordinate choice
A convenient choice is to track positions $r_i(t)$ and velocities $\dot r_i(t)$. The model then uses a kinetic term, a potential term, and a total action built from the time integral of the Lagrangian.

## Model-specific equations
- $T = \frac{1}{2} \sum_i m_i \dot r_i^2$
- $V = -G \sum_{i<j} \frac{m_i m_j}{\|r_i-r_j\|}$
- $L = T - V$
- $\ddot r_i = -G \sum_{j\ne i} m_j \frac{r_i-r_j}{\|r_i-r_j\|^3}$

## Derivation sketch
Differentiate the Lagrangian with respect to each coordinate and velocity, then apply the Euler-Lagrange equations to recover the Newtonian equations of motion.

## Simulation interpretation
The notebook compares the derived equations with a numerical simulation so that the reader can see how the theory and the computed trajectories fit together.

## Model Background and Notation

We model an observable $y(t)$ with a low-order mechanistic surrogate compatible with
Lagrangian/Hamiltonian intuition (energy exchange, damping, forcing, or transport).

General templates used in this notebook:
- Damped oscillator: $y(t)=A e^{-\gamma t}\cos(\omega t+\phi)+c$
- Logistic growth/saturation: $y(t)=\frac{K}{1+e^{-r(t-t_0)}}+c$
- Diffusion-like profile: $y(t)=A e^{-D t}+c$
- Michaelis-Menten-style response: $y(t)=\frac{V_{\max} t}{K_m+t}+c$

This project uses the **oscillator** surrogate by default.


## Public Data Sources and Peer-Reviewed References

1. **NASA JPL Small-Body Database**
   - URL: https://ssd-api.jpl.nasa.gov/cad.api?date-min=2026-01-01&date-max=2026-12-31&dist-max=0.05
   - Peer-reviewed context: Giorgini et al., Icarus (1996)
2. **Plotly Public Solar Dataset (CSV fallback)**
   - URL: https://raw.githubusercontent.com/plotly/datasets/master/solar.csv
   - Peer-reviewed context: Public benchmark fallback

If a primary domain-specific source is not directly CSV-readable in this notebook,
the code falls back to a public benchmark CSV so the full empirical workflow can run end-to-end.


In [1]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import requests
import ipywidgets as widgets
from ipywidgets import interact

plt.style.use('seaborn-v0_8-whitegrid')

DOMAIN_KEY = '01_celestial_mechanics'
DOMAIN_TITLE = 'Celestial Mechanics and Star-Chart Predictors'
MODEL_FAMILY = 'oscillator'
DATA_SOURCES = [
  {
    "name": "NASA JPL Small-Body Database",
    "url": "https://ssd-api.jpl.nasa.gov/cad.api?date-min=2026-01-01&date-max=2026-12-31&dist-max=0.05",
    "reference": "Giorgini et al., Icarus (1996)"
  },
  {
    "name": "Plotly Public Solar Dataset (CSV fallback)",
    "url": "https://raw.githubusercontent.com/plotly/datasets/master/solar.csv",
    "reference": "Public benchmark fallback"
  }
]


ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
def try_read_public_csv(url: str):
    try:
        r = requests.get(url, timeout=20)
        r.raise_for_status()
        if 'text/csv' in r.headers.get('Content-Type', '') or ',' in r.text[:2000]:
            return pd.read_csv(io.StringIO(r.text))
    except Exception:
        return None
    return None

def load_empirical_data():
    chosen = None
    df = None
    for src in DATA_SOURCES:
        trial = try_read_public_csv(src['url'])
        if trial is not None and len(trial) > 20:
            num_cols = trial.select_dtypes(include=[np.number]).columns.tolist()
            if len(num_cols) >= 1:
                df = trial.copy()
                chosen = src
                break

    if df is None:
        raise RuntimeError('No CSV-readable source found. Update DATA_SOURCES with a CSV endpoint.')

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_cols) == 1:
        x = np.arange(len(df), dtype=float)
        y = df[numeric_cols[0]].to_numpy(dtype=float)
    else:
        x = df[numeric_cols[0]].to_numpy(dtype=float)
        y = df[numeric_cols[1]].to_numpy(dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if len(x) > 2000:
        step = max(1, len(x) // 2000)
        x = x[::step]
        y = y[::step]

    if np.ptp(x) == 0:
        x = np.arange(len(x), dtype=float)

    x = (x - np.min(x)) / max(np.ptp(x), 1e-12)
    return x, y, chosen, df

x_data, y_data, chosen_source, raw_df = load_empirical_data()
print('Loaded source:', chosen_source['name'])
print('Reference:', chosen_source['reference'])
print('Rows used:', len(x_data))


In [ ]:
def model_function(x, p1, p2, p3, p4):
    if MODEL_FAMILY == 'oscillator':
        A, gamma, omega, c = p1, p2, p3, p4
        return A * np.exp(-gamma * x) * np.cos(omega * x) + c
    if MODEL_FAMILY == 'damped':
        A, gamma, _, c = p1, p2, p3, p4
        return A * np.exp(-gamma * x) + c
    if MODEL_FAMILY == 'logistic':
        K, r, x0, c = p1, p2, p3, p4
        return K / (1.0 + np.exp(-r * (x - x0))) + c
    if MODEL_FAMILY == 'michaelis':
        vmax, km, _, c = p1, p2, p3, p4
        return vmax * x / (km + x + 1e-12) + c
    if MODEL_FAMILY == 'diffusion':
        A, D, _, c = p1, p2, p3, p4
        return A * np.exp(-D * x) + c
    return p1 * x + p2

p0 = [np.nanmax(y_data) - np.nanmin(y_data), 0.5, 6.0, np.nanmean(y_data)]
bounds = ([-1e6, 0.0, -1e3, -1e6], [1e6, 50.0, 1e3, 1e6])

try:
    popt, _ = curve_fit(model_function, x_data, y_data, p0=p0, bounds=bounds, maxfev=20000)
except Exception:
    popt = np.array(p0, dtype=float)

y_hat = model_function(x_data, *popt)
rmse = float(np.sqrt(np.mean((y_data - y_hat) ** 2)))
r2 = float(1.0 - np.sum((y_data - y_hat) ** 2) / max(np.sum((y_data - np.mean(y_data)) ** 2), 1e-12))

print('Fitted parameters:', popt)
print('RMSE:', rmse)
print('R^2:', r2)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(x_data, y_data, s=12, alpha=0.5, label='Empirical data')
ax.plot(x_data, y_hat, color='crimson', linewidth=2.0, label='Fitted surrogate model')
ax.set_title(f'{DOMAIN_TITLE}: empirical fit ({MODEL_FAMILY})')
ax.set_xlabel('Normalized independent variable')
ax.set_ylabel('Observed variable')
ax.legend()
plt.show()


## Interactive Simulation Controls

Use the sliders below to perturb model parameters and observe how well the
simulation matches the empirical measurements.


In [ ]:
def interactive_plot(p1, p2, p3, p4):
    y_sim = model_function(x_data, p1, p2, p3, p4)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.scatter(x_data, y_data, s=10, alpha=0.45, label='Empirical data')
    ax.plot(x_data, y_sim, color='navy', linewidth=2.0, label='Simulation')
    ax.set_title(f'{DOMAIN_TITLE}: interactive parameter sweep')
    ax.set_xlabel('Normalized independent variable')
    ax.set_ylabel('Observed variable')
    ax.legend()
    plt.show()

interact(
    interactive_plot,
    p1=widgets.FloatSlider(value=float(popt[0]), min=-2*abs(float(popt[0]))-1, max=2*abs(float(popt[0]))+1, step=0.01),
    p2=widgets.FloatSlider(value=float(popt[1]), min=0.0, max=max(1.0, 3*abs(float(popt[1]))+1), step=0.01),
    p3=widgets.FloatSlider(value=float(popt[2]), min=-20.0, max=20.0, step=0.05),
    p4=widgets.FloatSlider(value=float(popt[3]), min=float(np.min(y_data))-2, max=float(np.max(y_data))+2, step=0.01),
);
